# Домашнее задание 3. Статистическая проверка качества регрессии.

### 1. Загрузите файл candidates.csv с компьютера. 

* откройте первые несколько строк
* изучите строение файла.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t, f
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
data = pd.read_csv("candidates.csv", sep=";", decimal=",")
display(data.head())
data.info()
print("Пропуски:", data.isna().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Код претендента                 800 non-null    int64  
 1   Рейтинг университета окончания  800 non-null    int64  
 2   Средний балл диплома            800 non-null    float64
 3   Опыт работы (лет)               800 non-null    int64  
 4   Возраст                         800 non-null    int64  
 5   Рейтинг университета диплома    800 non-null    int64  
 6   Баллы за сертификаты            800 non-null    int64  
 7   Желаемая заработная плата       800 non-null    int64  
 8   Итоговая зарплата от компании   800 non-null    int64  
dtypes: float64(1), int64(8)
memory usage: 56.4 KB
Пропуски: 0


,Код претендента,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата,Итоговая зарплата от компании
0,1001,82,3.22,10,33,84,89,237000,237000
1,1002,4,3.19,3,29,16,0,155000,155000
2,1003,70,3.84,6,30,70,74,212000,197000
3,1004,98,3.67,2,24,97,64,140000,130000
4,1005,94,3.92,2,27,99,66,146000,136000


### 2. На основе файла создайте матрицу X с характеристиками (некоторые столбцы могут не понадобятся для характеристик, подумайте какие), и столбец y (из последнего столбца).

* отобразите первые строки матрицы X и столбца y на экране.

In [2]:
X = data.drop(columns=["Код претендента", data.columns[-1]])
y = data.iloc[:, -1]
X_all = X.copy()
display(X.head())
display(y.head())

,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата
0,82,3.22,10,33,84,89,237000
1,4,3.19,3,29,16,0,155000
2,70,3.84,6,30,70,74,212000
3,98,3.67,2,24,97,64,140000
4,94,3.92,2,27,99,66,146000


0    237000
1    155000
2    197000
3    130000
4    136000
Name: Итоговая зарплата от компании, dtype: int64

Код претендента — это идентификатор, поэтому его не используем. Ответ — итоговая зарплата от компании. Остальные столбцы оставляем как признаки.

### 3. Используя SAG, постройте на этих данных модель линейной регрессии

* выведите на экран полученное уравнение линиии

In [3]:
def fit_sag(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model = Ridge(alpha=0.0, solver="sag", tol=1e-8,
                  max_iter=100000, random_state=42)
    model.fit(X_scaled, y)
    # Возвращаем коэффициенты к исходным единицам признаков.
    weights = model.coef_ / scaler.scale_
    intercept = model.intercept_ - weights @ scaler.mean_
    return np.r_[intercept, weights]


def predict(X, b):
    return b[0] + X.to_numpy() @ b[1:]


def show_equation(X, b):
    equation = f"Зарплата = {b[0]:.4f}"
    for name, weight in zip(X.columns, b[1:]):
        equation += f" {weight:+.6f} * [{name}]"
    print(equation)


b = fit_sag(X, y)
y_predict = predict(X, b)
show_equation(X, b)

Зарплата = 36695.8877 -154.744815 * [Рейтинг университета окончания] +11669.838536 * [Средний балл диплома] +9648.406165 * [Опыт работы (лет)] +64.994722 * [Возраст] -199.401570 * [Рейтинг университета диплома] +230.716349 * [Баллы за сертификаты] +0.326216 * [Желаемая заработная плата]


### 4. Проверьте веса на значимость, используя алгоритм и формулы из лекции

* код должен соотвествовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы
* какие веса у вас оказались незначимыми?

In [4]:
alpha = 0.05


def check_weights(X, y, b):
    n, k = X.shape
    df = n - k - 1
    X_matrix = np.column_stack([np.ones(n), X.to_numpy()])
    residuals = y.to_numpy() - predict(X, b)
    s2 = np.sum(residuals ** 2) / df
    C = s2 * np.linalg.inv(X_matrix.T @ X_matrix)
    standard_errors = np.sqrt(np.diag(C))
    t_values = b / standard_errors
    t_left = t.ppf(alpha / 2, df=df)
    t_right = t.ppf(1 - alpha / 2, df=df)
    significant = (t_values < t_left) | (t_values > t_right)
    table = pd.DataFrame({
        "Коэффициент": b,
        "Стандартная ошибка": standard_errors,
        "t": t_values,
        "p-value": 2 * t.sf(np.abs(t_values), df=df),
        "Значим": significant
    }, index=["Свободный член"] + X.columns.tolist())
    return table, df, t_left, t_right


weights_table, df, t_left, t_right = check_weights(X, y, b)
print("Степени свободы n - k - 1:", df)
print("Критический интервал:", t_left, t_right)
display(weights_table)
insignificant = weights_table.iloc[1:].index[~weights_table.iloc[1:]["Значим"]].tolist()
print("Незначимые признаки:", insignificant)

Степени свободы n - k - 1: 792
Критический интервал: -1.9629637813443193 1.9629637813443193
Незначимые признаки: ['Возраст']


,Коэффициент,Стандартная ошибка,t,p-value,Значим
Свободный член,36695.887727,5893.967199,6.226008,7.755663e-10,True
Рейтинг университета окончания,-154.744815,43.050605,-3.594486,3.451405e-04,True
Средний балл диплома,11669.838536,589.661594,19.790739,3.862320e-71,True
Опыт работы (лет),9648.406165,308.520856,31.273108,1.896109e-140,True
Возраст,64.994722,217.693375,0.298561,7.653534e-01,False
Рейтинг университета диплома,-199.401570,42.266715,-4.717697,2.819165e-06,True
Баллы за сертификаты,230.716349,11.016895,20.942049,7.738205e-78,True
Желаемая заработная плата,0.326216,0.012699,25.688524,2.461604e-106,True


Мы проверяем две гипотезы, чтобы понять, влияет ли признак на итоговую зарплату или нет. Первая гипотеза H0 говорит, что коэффициент равен нулю и не влияет на ответ. Вторая гипотеза H1 говорит, что коэффициент не равен нулю и влияет на ответ. Незначимым оказался «Возраст». Это значит, что в нашей модели возраст не даёт достаточно новой информации о зарплате

### 5. Если какие-то веса оказались незначимыми, перепишите матрицу характеристик X без соответствующих столбцов, обучите заново модель, используя SAG

In [5]:
X = X.drop(columns=insignificant)
b = fit_sag(X, y)
y_predict = predict(X, b)
print("Оставшиеся признаки:", X.columns.tolist())
show_equation(X, b)

Оставшиеся признаки: ['Рейтинг университета окончания', 'Средний балл диплома', 'Опыт работы (лет)', 'Рейтинг университета диплома', 'Баллы за сертификаты', 'Желаемая заработная плата']
Зарплата = 38290.3064 -154.283840 * [Рейтинг университета окончания] +11670.337468 * [Средний балл диплома] +9714.973269 * [Опыт работы (лет)] -199.828829 * [Рейтинг университета диплома] +230.644483 * [Баллы за сертификаты] +0.326051 * [Желаемая заработная плата]


### 6. Проверьте модель на значимость по алгоритму из лекции

* код должен соответствовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы и что эти гипотезы собой подразумевают
* какая у вас получилась модель?
* подходит ли она для описания наших данных?

In [6]:
n, k = X.shape
SSE = np.sum((y.to_numpy() - y_predict) ** 2)
SST = np.sum((y.to_numpy() - y.mean()) ** 2)
R2 = 1 - SSE / SST
F = (R2 / k) / ((1 - R2) / (n - k - 1))
F_critical = f.ppf(1 - alpha, dfn=k, dfd=n - k - 1)
p_value_F = f.sf(F, dfn=k, dfd=n - k - 1)
print("Степени свободы:", k, n - k - 1)
print("F:", F)
print("F критическое:", F_critical)
print("p-value:", p_value_F)
print("Модель значима:", F > F_critical)

Степени свободы: 6 793
F: 9183.214570072674
F критическое: 2.1099952841075096
p-value: 0.0
Модель значима: True


Мы проверяем две гипотезы, чтобы понять, влияет ли признак на итоговую зарплату или нет. Первая гипотеза H0 говорит, что коэффициент равен нулю и не влияет на ответ. Вторая гипотеза H1 говорит, что коэффициент не равен нулю и влияет на ответ. 
У нас модель получилась значимой, потому что F больше критического значения. Значит, признаки в модели действительно помогают предсказывать зарплату. 
Я думаю, что модель подходит для этих данных, потому что у неё высокий R2 и небольшая ошибка LOOCV по сравнению со средней зарплатой.

### 7. Рассчитайте парный коэффициент корреляции для каждой пары x_i, y, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* ответьте на вопрос "что в общем случае (а не в конкретно нашем) показывают парные коэффициенты корреляции?"
* выведите коэффициент корреляции на экран для каждой пары в виде: пара, значение коэффициента
* ответьте на вопрос "что полученные значения говорят о нашей модели?"
* какие характеристики оказывают на ответ наибольшее значение, какие наименьшее?

In [7]:
# Рассматриваем все исходные признаки, включая удалённый возраст.
correlations = []
y_centered = y.to_numpy() - y.mean()
sigma_y = np.sqrt(np.mean(y_centered ** 2))
for name in X_all.columns:
    x = X_all[name].to_numpy()
    x_centered = x - x.mean()
    sigma_x = np.sqrt(np.mean(x_centered ** 2))
    r = np.mean(x_centered * y_centered) / (sigma_x * sigma_y)
    correlations.append({"Признак": name, "r": r})
    print(f"{name} — зарплата: {r:.4f}")
correlations = pd.DataFrame(correlations)
display(correlations.sort_values("r", key=lambda s: s.abs(), ascending=False))

Рейтинг университета окончания — зарплата: -0.2725
Средний балл диплома — зарплата: 0.1372
Опыт работы (лет) — зарплата: 0.9418
Возраст — зарплата: 0.8907
Рейтинг университета диплома — зарплата: -0.2584
Баллы за сертификаты — зарплата: 0.1714
Желаемая заработная плата — зарплата: 0.9738


,Признак,r
6,Желаемая заработная плата,0.973753
2,Опыт работы (лет),0.941768
3,Возраст,0.890657
0,Рейтинг университета окончания,-0.272495
4,Рейтинг университета диплома,-0.258448
5,Баллы за сертификаты,0.171400
1,Средний балл диплома,0.137158


Парный коэффициент корреляции показывает, насколько два признака связаны между собой. Если он близок к 1, то при росте одного значения обычно растёт и другое. Если близок к -1, то одно растёт, а другое уменьшается. Если близок к нулю, то линейная связь слабая.

Самая сильная связь с итоговой зарплатой получилась у желаемой зарплаты и опыта работы. Самая слабая связь — у среднего балла диплома и баллов за сертификаты.

Это значит, что желаемая зарплата и опыт работы сильнее связаны с итоговой зарплатой. Но одна корреляция ещё не доказывает, что именно этот признак является причиной зарплаты.

### 8. Вычислите коэффициент детерминации

* код должен соответствовать формуле из лекции
* выведите полученный коэффициент на экран
* что он говорит о нашей модели? насколько качественно она описала данные? лучше или хуже она, чем средняя линия?

In [8]:
R2 = 1 - np.sum((y.to_numpy() - y_predict) ** 2) / np.sum((y.to_numpy() - y.mean()) ** 2)
rmse_train = np.sqrt(np.mean((y.to_numpy() - y_predict) ** 2))
print(f"R² = {R2:.6f}")
print(f"Объяснённая доля разброса: {R2 * 100:.2f}%")
print(f"RMSE на обучении: {rmse_train:.2f}")

R² = 0.985812
Объяснённая доля разброса: 98.58%
RMSE на обучении: 8655.16


R2 показывает, насколько модель лучше, чем простой прогноз средней зарплаты для всех кандидатов. У нас R2 примерно 0,986. Значит, модель очень хорошо описывает данные и намного лучше средней линии.

### 9. Рассчитайте LOOCV кросс-валидации, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* объясните устно как он работает и зачем он нужен
* выведите LOOCV на экран
* ответьте на вопрос "что полученный LOOCV говорит о нашей модели? она идеально нам подходит, или в ней нужно что-то менять?"

In [9]:
errors = []
mean_errors = []

for i in range(len(X_all)):
    mask = np.arange(len(X_all)) != i
    X_train = X_all.iloc[mask]
    y_train = y.iloc[mask]
    X_test = X_all.iloc[[i]]
    y_test = y.iloc[i]

    # Отбор признаков тоже повторяем без оставленного наблюдения.
    b_full = fit_sag(X_train, y_train)
    table, _, _, _ = check_weights(X_train, y_train, b_full)
    selected = table.iloc[1:].index[table.iloc[1:]["Значим"]].tolist()
    b_fold = fit_sag(X_train[selected], y_train)
    prediction = predict(X_test[selected], b_fold)[0]
    errors.append(y_test - prediction)
    mean_errors.append(y_test - y_train.mean())

LOOCV = np.sqrt(np.mean(np.array(errors) ** 2))
LOOCV_mean = np.sqrt(np.mean(np.array(mean_errors) ** 2))
print(f"LOOCV (RMSE): {LOOCV:.2f}")
print(f"LOOCV для прогноза средним: {LOOCV_mean:.2f}")
print(f"Средняя зарплата: {y.mean():.2f}")
print(f"LOOCV / средняя зарплата: {LOOCV / y.mean() * 100:.2f}%")

LOOCV (RMSE): 8739.41
LOOCV для прогноза средним: 72754.11
Средняя зарплата: 229032.50
LOOCV / средняя зарплата: 3.82%


Работа LOOCV выглядит так: мы по очереди убираем одного кандидата, обучаем модель на остальных и предсказываем зарплату убранного кандидата. Потом так делаем для каждого кандидата и считаем общую ошибку.

LOOCV нужен, чтобы проверить, как модель будет работать на новых данных.

У нас LOOCV примерно 8739. Это намного меньше ошибки простого прогноза средней зарплатой. Значит, модель в целом хорошая, но не идеальная. Ошибки всё равно есть, поэтому модель можно улучшать.